# Purchased-scope audit

Run from the Signal repository root using its Python environment. This notebook checks the saved comparison and replays the reviewed real notices without network calls. The full archive replay uses the accompanying `replay.py` and `compare.py`. Counts describe the frozen source snapshot, not a guarantee of global recall.


In [ ]:
import json, sys
from pathlib import Path
from collections import Counter
ROOT = Path.cwd()
assert (ROOT / 'pipeline').is_dir(), 'Run with the repository root as the working directory'
AUDIT = ROOT / 'docs/audits/purchased-scope-2026-09-18'
summary = json.loads((AUDIT / 'summary.json').read_text())
summary['before'], summary['after'], summary['comparison']


In [ ]:
assert summary['before']['retained'] == summary['after']['retained'] == 74820
assert summary['comparison']['unchanged_source_and_translation_content']
removed = summary['comparison']['removed_published']['count']
{'published_snapshot': 2964, 'removed': removed, 'preserved': 2964 - removed, 'removed_share_percent': round(100 * removed / 2964, 2)}


In [ ]:
sys.path.insert(0, str(ROOT / 'pipeline'))
from anthrion_signal.config import load_config
from anthrion_signal.discovery import prefilter
from anthrion_signal.models import Signal
config = load_config(ROOT)
cases = json.loads((ROOT / 'pipeline/tests/fixtures/purchased_scope_review_2026_09_18.json').read_text(encoding='utf-8'))
for case in cases:
    s = Signal.model_validate(case['signal'])
    original = (s.title, s.description, s.content_hash)
    prefilter([s], config['company_profile'], config['search_terms'], config['capabilities'], {s.id: case.get('translation', {})})
    retained = s.prefilter_score >= 12 and not s.exclusion_reasons
    assert bool(retained) == (case['expected'] == 'retain'), s.id
    assert original == (s.title, s.description, s.content_hash), s.id
Counter(c['expected'] for c in cases)


## Full replay

Create separate Git worktrees for the baseline source commit recorded in `summary.json` and the revised code. Use the baseline worktree as `--data` for both calls. Run `python docs/audits/purchased-scope-2026-09-18/replay.py --code BASELINE --data SNAPSHOT --output baseline.jsonl.gz`, then repeat with the revised code and `after.jsonl.gz`. Put both output files beside `compare.py` in a separate scratch directory and run that comparator. It asserts identical IDs, hashes, original and translated text, classifications and URLs before reporting decision changes. `--workers 1` runs the same batch evaluator sequentially. No provider API is called.

The classifier comparison precedes publisher identity reconciliation. Award counts here count unique notices, while market-specific public award files may contain the same notice in several markets. Closed records remain closed even when their scope is retained.
